# 📝 RAG 검색 평가 과제 정답 — 지표 네 개부터 k 고르기까지 (강사용)

각 문제의 **모범답안 + 해설**입니다. 경로는 `../../day20_ReAct_멀티툴_에이전트/data/` 입니다.

- 이 과제는 **모델을 한 번도 부르지 않습니다** — 임베딩과 검색만 씁니다. 값이 재현되므로 자가채점이 숫자를 정확히 봅니다.
- **1~4번**(지표 구현)은 손으로 만든 결정적 입력으로 값을 정확히 채점합니다.
- **5~7번**은 채점 셀이 **검색을 다시 돌려** 문항별로 대조합니다 — 값을 옮겨 적는 것으로는 통과할 수 없습니다. 평균은 임베딩 모델 버전 차이를 감안해 **넉넉한 범위**로 봅니다.

## 1. 검색 평가 지표 — Hit@K

**배경**: 검색이 쓸 만한지 알려면 **재야** 합니다. 질문마다 검색 결과 상위 K개와 미리 정해 둔 정답 목록을 견주어 점수를 매기는 것이 검색 평가입니다. 1~4번에서 그 눈금 네 개를 하나씩 손으로 만듭니다.

**네 함수의 인자는 모두 같습니다.** `ranked` 는 검색 결과 조각 id 를 **1위부터 순서대로** 담은 리스트, `gold` 는 그 질문의 **정답 조각 id 리스트**, `k` 는 상위 몇 개까지 볼지입니다. 보는 값은 같고 **무엇을 세느냐만** 다릅니다.

**요구사항**:
- 함수 **`hit_at_k(ranked, gold, k) -> float`** 를 정의하세요(모델도 데이터도 쓰지 않는 순수 계산입니다).
- Hit@K 는 **맞혔나 못 맞혔나만** 봅니다. 상위 `k` 개 안에 정답이 **하나라도** 있으면 `1.0`, 하나도 없으면 `0.0` 을 돌려줍니다.
- 돌려주는 값은 **실수(float)** 입니다 — `True`/`False` 나 정수 `1`/`0` 이 아닙니다. 네 지표는 결국 **여러 질문에 걸쳐 평균을 내는 값**이라, 나머지 셋(소수가 나옵니다)과 자료형을 맞춰 둡니다.

**예시**

```
hit_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3)   -> 1.0
hit_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 1)   -> 0.0   (1위만 보면 정답이 없다)
```

<details><summary>힌트</summary>

```text
접근방법:
- 상위 k개만 잘라 정답 목록과 겹치는 것이 있는지 보면 된다.

세부구현:
1. 검색 결과를 앞에서 k개만 자른다.
2. 그중 하나라도 정답 목록에 들어 있으면 1.0, 아니면 0.0 을 반환한다.
```

</details>

In [ ]:
def hit_at_k(ranked, gold, k):
    """상위 k개 안에 정답이 하나라도 있으면 1.0, 하나도 없으면 0.0 을 돌려준다."""
    # any() 는 True/False 를 주므로 지문이 요구한 실수 1.0/0.0 으로 바꿔 돌려준다
    return 1.0 if any(r in gold for r in ranked[:k]) else 0.0


print(hit_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3))   # 2위에 정답이 있다 -> 1.0
print(hit_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 1))   # 1위만 보면 없다 -> 0.0

In [ ]:
# [자가채점]
assert hit_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3) == 1.0
assert hit_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 1) == 0.0   # k 밖의 정답은 세지 않는다
assert hit_at_k(['a', 'b'], ['z'], 2) == 0.0                  # 정답이 하나도 없을 때
assert hit_at_k(['a', 'b', 'c'], ['c', 'b'], 2) == 1.0        # 정답이 여럿이어도 하나만 맞으면 된다
assert type(hit_at_k(['a'], ['a'], 1)) is float, '1.0 또는 0.0(실수)을 돌려주세요'
print('✅ 통과!')

**해설**: Hit@K 는 네 지표 중 가장 무딘 눈금입니다 — **정답을 건졌나 못 건졌나**만 봅니다. 그래서 값이 높게 나오기 쉽고, 높다고 검색이 좋다는 뜻은 아닙니다.

**흔한 실수 두 가지**: (1) `any(...)` 결과를 그대로 돌려주면 `True`/`False` 라 타입 검사에서 걸립니다. (2) `ranked` 전체를 보면 `k` 밖에 있는 정답까지 세어 점수가 실제보다 높아집니다 — 반드시 앞에서 `k` 개만 자릅니다.

## 2. 검색 평가 지표 — Precision@K

**요구사항**: 함수 **`precision_at_k(ranked, gold, k) -> float`** 를 정의하세요. 인자는 1번과 같습니다.

- Precision@K 는 **내가 꺼내 온 것 중 몇 개가 정답이었나**를 봅니다.
- 상위 `k` 개 안에 든 정답의 개수를 세어 **`k` 로 나눈 값**을 돌려줍니다.
- 나누는 수는 검색 결과의 길이가 아니라 언제나 **`k`** 입니다. 결과가 `k` 개보다 적게 와도 `k` 로 나눕니다.

**예시**

```
precision_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3)   -> 0.333...   (3개 중 1개가 정답)
precision_at_k(['a', 'b', 'c'], ['b', 'c'], 3)        -> 0.666...   (3개 중 2개가 정답)
```

<details><summary>힌트</summary>

```text
접근방법:
- 상위 k개 중 정답 목록에 든 것의 개수를 세어 k로 나눈다.

세부구현:
1. 검색 결과를 앞에서 k개만 자른다.
2. 그중 정답 목록에 있는 것의 개수를 센다.
3. 그 개수를 k로 나눠 반환한다.
```

</details>

In [ ]:
def precision_at_k(ranked, gold, k):
    """상위 k개 중 정답의 비율(꺼내 온 것 중 몇 개가 정답이었나)을 돌려준다."""
    # 나누는 수는 언제나 k 다. len(ranked[:k]) 로 쓰면 결과가 k개보다 적을 때 값이 달라진다
    return sum(1 for r in ranked[:k] if r in gold) / k


print(precision_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3))   # 1/3
print(precision_at_k(['a', 'b', 'c'], ['b', 'c'], 3))        # 2/3

In [ ]:
# [자가채점]
# 나눗셈 결과라 부동소수 오차가 있을 수 있어 정확일치 대신 아주 작은 차이로 비교한다
assert abs(precision_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3) - 1 / 3) < 1e-9
assert abs(precision_at_k(['a', 'b', 'c'], ['b', 'c'], 3) - 2 / 3) < 1e-9
assert precision_at_k(['a', 'b', 'c'], ['z'], 3) == 0.0        # 정답이 하나도 없을 때
assert abs(precision_at_k(['b'], ['b'], 3) - 1 / 3) < 1e-9, '결과가 k개보다 적어도 k 로 나눠야 합니다'
print('✅ 통과!')

**해설**: Precision@K 는 **꺼내 온 것 중** 몇 개가 정답이었는지를 봅니다. 정답이 하나뿐인 질문은 K 를 키울수록 이 값이 내려갑니다 — 정답은 그대로인데 나누는 수만 커지기 때문입니다.

**흔한 실수**: `len(ranked[:k])` 로 나누면 검색 결과가 `k` 개보다 적을 때 점수가 부풀려집니다. 지표의 정의대로 **언제나 `k`** 로 나눕니다.

## 3. 검색 평가 지표 — Recall@K

**요구사항**: 함수 **`recall_at_k(ranked, gold, k) -> float`** 를 정의하세요. 인자는 1번과 같습니다.

- Recall@K 는 **찾았어야 할 정답 중 몇 개를 건졌나**를 봅니다.
- 상위 `k` 개 안에 든 정답의 개수를 세어 **그 질문의 전체 정답 개수(`gold` 의 길이)로 나눈 값**을 돌려줍니다.
- 세는 방법은 2번과 같고 **나누는 수만 다릅니다.**

**예시**

```
recall_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3)   -> 0.333...   (정답 3개 중 1개를 건졌다)
recall_at_k(['a', 'b', 'c'], ['b'], 3)             -> 1.0        (정답 1개를 다 건졌다)
```

<details><summary>힌트</summary>

```text
접근방법:
- 2번과 세는 방법은 같고 나누는 수만 다르다 — 이번에는 전체 정답 개수로 나눈다.

세부구현:
1. 검색 결과를 앞에서 k개만 자른다.
2. 그중 정답 목록에 있는 것의 개수를 센다.
3. 그 개수를 전체 정답 개수로 나눠 반환한다.
```

</details>

In [ ]:
def recall_at_k(ranked, gold, k):
    """상위 k개가 그 질문의 전체 정답 중 몇 할을 건졌는지 돌려준다."""
    # 세는 방법은 Precision 과 같고 나누는 수만 다르다
    # k 로 나누면 '꺼내 온 것 중', 전체 정답 수로 나누면 '찾았어야 할 것 중' 이 된다
    return sum(1 for r in ranked[:k] if r in gold) / len(gold)


print(recall_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3))   # 정답 3개 중 1개 -> 1/3
print(recall_at_k(['a', 'b', 'c'], ['b'], 3))             # 정답 1개를 다 건졌다 -> 1.0

In [ ]:
# [자가채점]
assert abs(recall_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3) - 1 / 3) < 1e-9
assert recall_at_k(['a', 'b', 'c'], ['b', 'x'], 3) == 0.5
assert recall_at_k(['a', 'b', 'c'], ['b'], 3) == 1.0
assert recall_at_k(['a', 'b', 'c'], ['z'], 3) == 0.0           # 정답이 하나도 없을 때
# 정답이 k 보다 많으면 아무리 잘 검색해도 1.0 이 될 수 없다
assert abs(recall_at_k(['a', 'b', 'c'], ['a', 'b', 'c', 'd'], 3) - 0.75) < 1e-9
print('✅ 통과!')

**해설**: Recall@K 는 **놓친 것이 있는지**를 봅니다. 정답이 `k` 개보다 많은 질문은 상위 `k` 개를 다 맞혀도 1.0 이 나올 수 없습니다 — 값이 낮다고 무조건 검색이 나쁜 것은 아니라는 뜻입니다.

**흔한 실수**: 나누는 수를 `k` 로 두면 Precision 과 똑같은 함수가 됩니다. 두 지표의 차이는 **분모 하나**뿐입니다.

## 4. 검색 평가 지표 — MRR@K

**요구사항**: 함수 **`mrr_at_k(ranked, gold, k) -> float`** 를 정의하세요. 인자는 1번과 같습니다.

- MRR@K 는 **정답이 얼마나 위쪽에 있었나**를 봅니다. 상위 `k` 개를 1위부터 보다가 **처음 만난 정답의 순위로 1 을 나눈 값**을 돌려줍니다(1위면 `1.0`, 2위면 `0.5`, 4위면 `0.25`).
- 정답을 두 개 이상 만나도 **처음 만난 것 하나만** 씁니다.
- 상위 `k` 개 안에 정답이 하나도 없으면 `0.0` 을 돌려줍니다. `k` 밖의 정답은 세지 않습니다.

**예시**

```
mrr_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3)   -> 0.5   (첫 정답이 2위)
mrr_at_k(['b', 'a', 'c'], ['b', 'x'], 3)        -> 1.0   (첫 정답이 1위)
```

<details><summary>힌트</summary>

```text
접근방법:
- 앞에서부터 순서대로 보다가 처음 만난 정답의 순위를 쓰면 된다.

세부구현:
1. 상위 k개를 순위 번호와 함께 앞에서부터 돈다(첫 번째가 1위).
2. 정답 목록에 있는 것을 처음 만나면 1을 그 순위로 나눠 반환한다.
3. 끝까지 못 만나면 0.0 을 반환한다.
```

</details>

In [ ]:
def mrr_at_k(ranked, gold, k):
    """처음 만난 정답의 순위 역수를 돌려준다(위쪽일수록 높다). 없으면 0.0."""
    # enumerate 의 두 번째 인자 1 : 순위는 0 이 아니라 1 부터 세기 때문
    for rank, r in enumerate(ranked[:k], 1):
        if r in gold:
            return 1 / rank      # 처음 만난 정답 하나만 쓰고 바로 끝낸다
    return 0.0                   # 상위 k개 안에 정답이 없었다


print(mrr_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3))   # 첫 정답이 2위 -> 0.5
print(mrr_at_k(['b', 'a', 'c'], ['b', 'x'], 3))        # 첫 정답이 1위 -> 1.0

In [ ]:
# [자가채점]
assert mrr_at_k(['a', 'b', 'c'], ['b', 'x', 'y'], 3) == 0.5
assert mrr_at_k(['b', 'a', 'c'], ['b', 'x'], 3) == 1.0        # 1위에 있으면 1.0
assert mrr_at_k(['a', 'b', 'c'], ['z'], 3) == 0.0             # 정답이 하나도 없을 때
assert mrr_at_k(['a', 'b', 'c'], ['c'], 2) == 0.0             # k 밖의 정답은 세지 않는다
assert mrr_at_k(['a', 'b', 'c'], ['b', 'c'], 3) == 0.5        # 정답이 여럿이어도 처음 하나만
print('✅ 통과!')

**해설**: MRR@K 만이 **순위**를 봅니다. 나머지 셋은 정답이 1위에 있든 3위에 있든 같은 점수를 주지만, MRR 은 위에 있을수록 높습니다 — 사용자가 맨 위부터 읽는다는 사실을 반영한 눈금입니다.

**흔한 실수 두 가지**: (1) 순위를 0부터 세면 첫 항목에서 0으로 나누게 됩니다. (2) 정답을 만나도 끝까지 돌며 계속 더하면 MRR 이 아니라 다른 값이 됩니다 — **처음 만난 하나**에서 멈춥니다.

## 5. 검색 품질 재기 — 평가셋 전체를 네 지표로
**배경**: 여기까지는 손으로 만든 짧은 리스트로 지표를 확인했습니다. 이제 **진짜 검색기**에 붙입니다. 아래 평가셋에는 질문 25개마다 **정답 조각 id** 가 붙어 있습니다. 문항마다 검색을 하고 1~4번에서 만든 네 지표를 계산해 평균을 내면 이 검색기의 **성적표**가 됩니다.

> 이 문제의 코퍼스는 교안이 쓴 안내서가 아니라 **개인정보 질의응답 모음집**입니다. 아래 **제공 셀 세 개**(평가 코퍼스 살펴보기 → 청킹 → 색인)를 위에서부터 실행한 뒤 푸세요. 검색은 **`search_ids(질문, k)`** 로 하고, 조각 id 를 1위부터 순서대로 돌려줍니다. 지표는 1~4번에서 만든 **`hit_at_k`·`precision_at_k`·`recall_at_k`·`mrr_at_k`** 를 그대로 씁니다(다시 만들지 마세요 — 인자는 그때와 같은 `(ranked, gold, k)` 입니다).

**요구사항**: `K = 3` 으로 평가셋 전체를 재서 세 가지를 만드세요.

- 문항마다 상위 3개 조각 id 를 얻습니다. 검색은 문항당 **한 번만** 하고, 그 결과 하나로 네 지표를 모두 계산합니다.
- 정답 라벨 `gold_chunks` 는 `'|'` 로 이어져 있으니 **나눠서 리스트로** 만들어 넘깁니다.
- 문항별 결과를 DataFrame **`eval_results`** 로 만드세요. 열은 **`query_id`, `Hit`, `P`, `R`, `MRR`** 다섯 개이고 **이 순서**여야 합니다. 한 행이 한 문항이므로 행 수는 평가 문항 수와 같습니다.
- 네 지표 각각의 평균을 딕셔너리 **`eval_summary`** 에 담으세요. 열쇠는 **`'Hit'`, `'P'`, `'R'`, `'MRR'`** 네 개입니다.
- `Hit` 이 `0.0` 인 문항의 `query_id` 를 리스트 **`missed_qna`** 에 담으세요(평가셋에 나온 순서 그대로).

**예시**: 제대로 재면 평균은 **Hit 약 0.96 · P 약 0.53 · R 약 0.72 · MRR 약 0.90** 근처가 나오고, `missed_qna` 에는 **문항 하나**만 남습니다 — 어느 질문이 걸렸는지 직접 열어 보세요. 환경에 따라 값이 조금 다를 수 있어 채점은 **넉넉한 범위**로 봅니다(값을 정확히 맞힐 필요는 없습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 문항을 한 번씩 돌면서 검색을 한 번만 하고, 그 결과 목록으로 네 지표를 모두 계산한다.

세부구현:
1. 빈 목록을 만들어 두고 평가셋을 한 행씩 돈다.
   1-1. 질문으로 상위 3개 조각 id 를 얻는다.
   1-2. 정답 라벨을 구분자로 나눠 목록으로 만든다.
   1-3. 네 지표를 계산해 딕셔너리 하나로 담아 목록에 넣는다(적은 순서가 곧 열 순서다).
2. 목록으로 DataFrame 을 만든다.
3. 네 열의 평균을 딕셔너리로 만든다.
4. Hit 이 0 인 행만 골라 그 문항 번호를 목록으로 뽑는다.
```

</details>

In [ ]:
# [제공 코드] 평가 코퍼스와 평가셋을 읽고 눈으로 확인합니다 — 이 셀은 실행만 하세요.
import pandas as pd

# 문서 모음 — 한 행이 문서 하나이고, 검색 대상 글은 '본문' 열에 있습니다.
qna_df = pd.read_csv('../../day20_ReAct_멀티툴_에이전트/data/qna_docs.csv')
# 평가셋 — 질문(query)마다 정답 조각 id(gold_chunks)가 '|' 로 이어져 붙어 있습니다.
evalset = pd.read_csv('../../day20_ReAct_멀티툴_에이전트/data/qna_eval_chunk.csv')

print('문서', len(qna_df), '건 / 평가 문항', len(evalset), '건')
display(qna_df[['id', '분야', '질문']].head(3))
display(evalset[['query_id', 'query', 'gold_chunks', '유형']].head(3))

In [ ]:
# 청킹 — 19일차에서 배운 그 스플리터입니다(문단 -> 줄 -> 문장 -> 낱말 순으로 큰 경계부터 존중합니다).
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size=400 : 안내서 한 쪽이 길어 반드시 잘립니다. chunk_overlap=80 은 경계에서 문장이 반 토막 나는 것을 막아 줍니다.
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
print('스플리터 준비 완료')

In [ ]:
# [제공 코드] 문서를 조각으로 나눠 색인하고 검색 함수를 만듭니다 — 이 셀은 실행만 하세요.
#  (임베딩 모델을 내려받느라 처음 한 번은 잠시 걸립니다.)
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

# 조각 id 는 '문서id-순번' 규칙입니다. 평가셋의 정답 라벨이 이 규칙으로 붙어 있으므로
#  청킹 기준이나 id 규칙을 바꾸면 정답과 어긋나 점수가 전부 달라집니다.
qna_chunks, qna_ids = [], []
for doc_id, text in zip(qna_df['id'], qna_df['본문']):
    for i, part in enumerate(splitter.split_text(text)):
        qna_chunks.append(Document(page_content=part))
        qna_ids.append(f'{doc_id}-{i}')

qna_embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')
# ids 를 함께 넘기면 같은 id 는 덮어쓰기가 됩니다 -> 이 셀을 여러 번 실행해도 조각이 중복되지 않습니다.
#  ids 로 넘긴 값이 곧 그 조각의 열쇠이고, 검색 결과의 Document.id 로 그대로 돌아옵니다.
qna_store = Chroma.from_documents(qna_chunks, qna_embeddings, collection_name='qna_eval',
                                  ids=qna_ids)


def search_ids(query, k):
    """질문과 의미가 가까운 조각 id 를 1위부터 k개까지 순서대로 돌려준다."""
    # 평가에서는 k 를 바꿔 가며 재야 해서, k 를 인자로 받는 검색을 그대로 씁니다.
    return [d.id for d in qna_store.similarity_search(query, k=k)]


print('조각', len(qna_chunks), '개 색인 완료')
print('첫 문항 검색 결과:', search_ids(evalset['query'].iloc[0], 3))

In [ ]:
# 문항마다 검색은 한 번만 한다 — 같은 질문을 지표 개수만큼 네 번 검색할 이유가 없다
rows = []
for query_id, query, gold_text in zip(evalset['query_id'], evalset['query'], evalset['gold_chunks']):
    ranked = search_ids(query, 3)
    gold = gold_text.split('|')      # 정답이 여러 개인 문항은 '|' 로 이어져 있다
    # 딕셔너리에 적어 둔 순서가 그대로 DataFrame 의 열 순서가 된다
    rows.append({'query_id': query_id,
                 'Hit': hit_at_k(ranked, gold, 3),
                 'P': precision_at_k(ranked, gold, 3),
                 'R': recall_at_k(ranked, gold, 3),
                 'MRR': mrr_at_k(ranked, gold, 3)})

eval_results = pd.DataFrame(rows)
# 네 지표의 평균 -- 이 검색기의 성적표다
eval_summary = {m: eval_results[m].mean() for m in ['Hit', 'P', 'R', 'MRR']}
# Hit 이 0 인 문항 = 상위 3개 안에 정답이 하나도 없던 질문. 검색기를 고칠 실마리가 여기 있다
missed_qna = eval_results.loc[eval_results['Hit'] == 0.0, 'query_id'].tolist()

display(eval_results.round(3))
print(eval_summary)
print('놓친 문항:', missed_qna)

In [ ]:
# [자가채점]
# 값을 손으로 적어 넣으면 통과하지 못하도록, 채점이 검색을 다시 돌려 정답을 스스로 계산한다
# (임베딩·검색은 결정적이라 같은 질문·같은 k 면 결과가 항상 같다)
assert list(eval_results.columns) == ['query_id', 'Hit', 'P', 'R', 'MRR'], '열 이름과 순서를 확인하세요'
assert len(eval_results) == len(evalset), '모든 문항을 재야 합니다'

for query_id, query, gold_text in zip(evalset['query_id'], evalset['query'], evalset['gold_chunks']):
    ranked = search_ids(query, 3)
    gold = gold_text.split('|')
    mine = eval_results[eval_results['query_id'] == query_id].iloc[0]
    assert mine['Hit'] == hit_at_k(ranked, gold, 3), f'{query_id} Hit'
    assert abs(mine['P'] - precision_at_k(ranked, gold, 3)) < 1e-9, f'{query_id} P'
    assert abs(mine['R'] - recall_at_k(ranked, gold, 3)) < 1e-9, f'{query_id} R'
    assert abs(mine['MRR'] - mrr_at_k(ranked, gold, 3)) < 1e-9, f'{query_id} MRR'

# 평균은 값을 정확히 맞히는 대신 '이 언저리인지'만 본다
#  (임베딩 모델 버전이 달라 한두 문항이 뒤집혀도 옳은 풀이는 통과하고,
#   분모를 잘못 쓴 계산은 다른 지표 값으로 넘어가 버리므로 범위 밖으로 떨어진다)
for metric, low, high in [('Hit', 0.85, 1.00), ('P', 0.45, 0.65),
                          ('R', 0.65, 0.85), ('MRR', 0.85, 1.00)]:
    assert abs(eval_summary[metric] - eval_results[metric].mean()) < 1e-9, f'eval_summary[{metric}] 가 다릅니다'
    assert low < eval_summary[metric] <= high, f'{metric} 평균이 지문에 적힌 언저리를 벗어났습니다'

assert missed_qna == eval_results.loc[eval_results['Hit'] == 0.0, 'query_id'].tolist()
assert len(missed_qna) == 1, '상위 3개 안에 정답이 하나도 없는 문항은 한 건입니다'
print('✅ 통과!')

**해설**: 검색기를 고치려면 먼저 **재야** 합니다. 문항 25개를 한 번씩 검색해 네 눈금을 매기고 평균을 내면, '좋아 보인다'가 **Hit 0.96 · P 0.53 · R 0.72 · MRR 0.90** 라는 비교 가능한 숫자가 됩니다. 다음에 무엇을 바꾸든 이 숫자와 견주면 나아졌는지 알 수 있습니다.

**표를 읽는 법**: Hit 가 0.96 인데 P 가 0.53 인 것은 모순이 아닙니다. Hit 는 '하나라도 맞혔나', P 는 '꺼내 온 3개 중 몇 개가 정답인가'라 **보는 것이 다릅니다**. 정답이 1개뿐인 문항은 완벽하게 검색해도 P 가 1/3 입니다.

**놓친 문항이 진짜 소득**: 평균보다 `missed_qna` 이 값집니다. 상위 3개 안에 정답이 하나도 못 들어온 질문 하나를 열어 보면, 그 질문이 문서와 **어떤 낱말도 겹치지 않는지**·정답 조각이 너무 길거나 짧은지 같은 원인이 보입니다.

**채점 기준**: 값을 옮겨 적는 것으로는 통과할 수 없게, 채점 셀이 **검색을 다시 돌려** 문항별로 대조합니다(여기는 오차 없이 그대로 일치해야 합니다). 반면 **평균 네 개는 범위로** 봅니다 — 학생이 1~4번에서 만든 함수로 재는 것이라, 분모를 잘못 쓴 계산은 이 범위 밖으로 떨어집니다.

## 6. K 를 바꿔 가며 비교 — 넓게 볼수록 좋을까
**배경**: 5번에서는 K 를 3 으로 고정했습니다. K 는 **상위 몇 개를 채택할지 정하는 파라미터**입니다. 어느 값이 나은지는 **재 봐야** 압니다.

**요구사항**: `K = 1, 3, 5, 10` 네 가지로 각각 평가셋 전체를 재서 DataFrame **`k_table`** 을 만드세요.

- 열은 **`K`, `Hit`, `P`, `R`, `MRR`** 다섯 개이고 **이 순서**입니다. 한 행이 K 하나이고, 행 순서는 **1, 3, 5, 10** 입니다.
- 각 칸은 그 K 로 잰 **전 문항 평균**입니다(5번과 같은 방법, K 만 바꿉니다). 지표는 5번과 마찬가지로 1~4번의 **`hit_at_k`·`precision_at_k`·`recall_at_k`·`mrr_at_k`** 를 씁니다.
- 검색은 문항마다 **가장 큰 K(=10)로 한 번만** 하고, 그 목록 하나로 네 K 를 모두 계산하세요. 지표 함수가 알아서 앞에서 `k` 개만 보므로 목록을 다시 자를 필요가 없습니다.

**예시**: 위의 두 행은 대략 이렇게 나옵니다(소수 셋째 자리까지 — 환경에 따라 조금 다를 수 있어 채점은 **넉넉한 범위와 방향**으로 봅니다).

```
 K    Hit      P      R    MRR
 1  0.840  0.840  0.435  0.840
 3  0.960  0.533  0.717  0.900
 5    ...    ...    ...    ...
10    ...    ...    ...    ...
```

표를 다 채우면 **K 를 키울 때 오르는 지표와 내려가는 지표가 갈립니다.** 그 모습을 확인하고 아래 서술 답안을 채우세요.

<details><summary>힌트</summary>

```text
접근방법:
- 문항마다 상위 10개를 한 번 받아 두고, 같은 목록에 k 만 바꿔 넣어 네 번 잰다.

세부구현:
1. 평가셋을 한 번 돌며 (상위 10개 조각 id, 정답 목록) 짝을 모아 둔다.
2. K 후보 네 개를 차례로 돈다.
   2-1. 모아 둔 짝마다 네 지표를 계산해 문항 수로 나눠 평균을 낸다.
   2-2. K 와 네 평균을 딕셔너리 하나로 담아 목록에 넣는다.
3. 목록으로 DataFrame 을 만든다.
```

</details>

In [ ]:
# 검색은 문항마다 한 번만 -- 가장 큰 K(10)로 받아 두면 작은 K 는 지표 함수가 앞에서 잘라 쓴다
top10 = [(search_ids(query, 10), gold_text.split('|'))
         for query, gold_text in zip(evalset['query'], evalset['gold_chunks'])]

k_rows = []
for k in [1, 3, 5, 10]:
    # 같은 검색 결과 목록에 k 만 바꿔 넘기면 그 K 의 점수가 나온다
    k_rows.append({'K': k,
                   'Hit': sum(hit_at_k(r, g, k) for r, g in top10) / len(top10),
                   'P': sum(precision_at_k(r, g, k) for r, g in top10) / len(top10),
                   'R': sum(recall_at_k(r, g, k) for r, g in top10) / len(top10),
                   'MRR': sum(mrr_at_k(r, g, k) for r, g in top10) / len(top10)})

k_table = pd.DataFrame(k_rows)
display(k_table.round(3))

In [ ]:
# [자가채점]
assert list(k_table.columns) == ['K', 'Hit', 'P', 'R', 'MRR'], '열 이름과 순서를 확인하세요'
assert k_table['K'].tolist() == [1, 3, 5, 10], 'K 는 1, 3, 5, 10 순서여야 합니다'

# 채점도 같은 방식으로 다시 재서 대조한다 — 표에 값을 적어 넣는 것으로는 통과하지 못한다
pairs = [(search_ids(query, 10), gold_text.split('|'))
         for query, gold_text in zip(evalset['query'], evalset['gold_chunks'])]
for i, k in enumerate([1, 3, 5, 10]):
    for metric, metric_fn in [('Hit', hit_at_k), ('P', precision_at_k),
                              ('R', recall_at_k), ('MRR', mrr_at_k)]:
        want = sum(metric_fn(r, g, k) for r, g in pairs) / len(pairs)
        assert abs(k_table.loc[i, metric] - want) < 1e-9, f'K={k} 의 {metric} 값이 다릅니다'

# 지문에 적어 둔 K=1 · K=3 행이 그 언저리인지 본다(정확한 값이 아니라 범위로)
assert 0.80 < k_table.loc[0, 'Hit'] <= 1.00, 'K=1 의 Hit 가 지문의 언저리를 벗어났습니다'
assert 0.35 < k_table.loc[0, 'R'] < 0.55, 'K=1 의 Recall 이 지문의 언저리를 벗어났습니다'
assert 0.45 < k_table.loc[1, 'P'] < 0.65, 'K=3 의 Precision 이 지문의 언저리를 벗어났습니다'

# K 를 키우면 Recall 은 오르고 Precision 은 떨어진다 -- 트레이드오프가 표에 그대로 보인다
assert k_table.loc[3, 'R'] > k_table.loc[0, 'R'], 'K 가 커지면 Recall 은 올라야 합니다'
assert k_table.loc[3, 'P'] < k_table.loc[0, 'P'], 'K 가 커지면 Precision 은 내려야 합니다'
print('✅ 통과!')

**서술 답안** — 표를 보고 아래에 적으세요.

*(여기에 이 검색기의 K 를 얼마로 정할지, 표의 어떤 값을 근거로 그렇게 정했는지 서술하세요)*

**해설**: K 는 **Precision 과 Recall 을 맞바꾸는 파라미터**입니다. 넓게 볼수록 정답을 놓치지 않지만(Recall 0.435 → 0.907), 꺼내 온 것 중 정답의 비율은 떨어집니다(Precision 0.840 → 0.224). 한쪽만 보고 "K 를 키우니 좋아졌다"고 말하면 절반만 본 것입니다.

**MRR 이 꿈쩍도 않는 이유**: K=3 에서 0.900, K=10 에서도 0.900 으로 똑같습니다. MRR 은 **처음 만난 정답의 순위**만 보므로, 이미 위쪽에서 찾은 문항은 K 를 키워도 점수가 그대로입니다. K 를 키워 새로 건지는 정답은 순위가 낮아 기여가 작습니다.

**모범 서술**: 이 검색기는 **K = 3** 이 무난합니다. K 를 1 에서 3 으로 키울 때는 Hit 가 0.840 에서 0.960 으로, Recall 이 0.435 에서 0.717 로 크게 올라 **얻는 것이 큽니다**. 그런데 3 에서 5 로 더 키워도 Hit 는 0.960 그대로이고 Recall 은 0.717 에서 0.757 로 거의 늘지 않는데 Precision 만 0.533 에서 0.352 로 떨어집니다 — **더 넣어 봐야 대부분 정답이 아닌 조각**이라는 뜻입니다.

다만 정답은 하나가 아닙니다. **검색 결과를 사람이 읽는** 화면이라면 위에서 몇 개만 보므로 K 를 작게 잡는 편이 낫고, **놓치면 큰일 나는 일**(예: 규정 위반 근거를 빠짐없이 모아야 할 때)이라면 Precision 을 내주더라도 K 를 10 으로 키워 Recall 0.907 을 택할 수 있습니다. 중요한 것은 **무엇을 잃고 무엇을 얻는지 표로 알고 고르는 것**입니다.

## 7. 검색기를 재서 `k` 를 정하기
**배경**: 마지막 문제는 **다른 검색기**를 재서 실제 서비스에 넣을 `k` 를 **근거를 가지고** 고릅니다. 서점 고객센터 FAQ 22건을 색인하고, 손으로 만든 평가셋 14문항으로 잽니다.

지금까지는 `k=2` 나 `k=3` 을 그냥 썼습니다. 이번에는 **왜 그 값인지**를 수치로 말할 수 있게 만듭니다.

제공 셀이 색인(`bs_store`)과 검색 함수(`bs_search_ids`), 평가셋(`bs_eval`)을 만들어 둡니다. 지표는 1~4번에서 만든 **`hit_at_k`·`precision_at_k`·`recall_at_k`·`mrr_at_k`** 를 그대로 씁니다 — 3단계에서 `hit_at_k` 로 못 찾은 문항을 가려낼 때도 같은 함수를 씁니다.

> 검색 함수와 표 이름이 `bs_` 로 시작합니다 — 위 5·6번이 만든 `search_ids`·`k_table`(질의응답 코퍼스)을 덮어쓰면 **에러 없이 엉뚱한 코퍼스를 재게** 되기 때문입니다.

In [ ]:
# [제공 코드] 서점 FAQ 색인과 평가셋 — 이 셀은 실행만 하세요(임베딩에 잠시 걸립니다).
#  FAQ 한 행이 조각 하나입니다(한 건이 짧아 자를 것이 없습니다) -> 조각 id 는 FAQ 의 id 그대로입니다.
#  검색 함수 이름이 bs_ 로 시작합니다 -- 위 문제들의 search_ids(질의응답 코퍼스)를 덮어쓰지 않기 위해서입니다.
import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

bs_faq = pd.read_csv('../../day20_ReAct_멀티툴_에이전트/data/bookstore_faq.csv')
bs_eval = pd.read_csv('../../day20_ReAct_멀티툴_에이전트/data/bookstore_eval.csv')

bs_documents = [Document(page_content=f'{r.title}\n{r.text}') for r in bs_faq.itertuples()]
bs_embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')
bs_store = Chroma.from_documents(bs_documents, bs_embeddings,
                                 collection_name='bookstore_eval', ids=list(bs_faq['id']))


def bs_search_ids(query, k):
    """질문과 가장 가까운 서점 FAQ k개의 id 를 순위 순서로 돌려준다."""
    return [d.id for d in bs_store.as_retriever(search_kwargs={'k': k}).invoke(query)]


print('FAQ', len(bs_faq), '건 색인 / 평가셋', len(bs_eval), '문항')
display(bs_eval.head(3))

### 1단계 — K 별로 재서 표 만들기

`K` 를 **1·2·3·5** 로 바꿔 가며 평가셋 **전체 평균**을 재고, 그 결과를 DataFrame **`bs_k_table`** 에 담으세요.

- 정답 라벨은 `bs_eval` 의 **`gold_chunks`** 열에 `'|'` 로 이어져 있습니다 — 나눠서 리스트로 쓰세요.
- 검색은 제공된 **`bs_search_ids(query, k)`** 를 쓰세요.
- `bs_k_table` 의 열은 **`['K', 'Hit', 'P', 'R', 'MRR']`** 이고 행은 K 오름차순 **4행**입니다.
- 각 지표 값은 그 K 로 잰 **14문항 평균**입니다.

**예시**: `bs_k_table` 의 첫 행은 `K=1` 이고, 그때 `Hit` 은 0.9 보다 작습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 바깥 반복은 K, 안쪽 반복은 문항이다. 문항마다 한 번만 검색해 네 지표를 모두 뽑는다.

세부구현:
1. K 값 네 개를 순서대로 돈다.
2. 그 안에서 평가셋의 행을 돌며
   2-1. 질문으로 상위 K개 id 를 찾고
   2-2. 정답 문자열을 구분자로 나눠 리스트로 만들고
   2-3. 네 지표를 각각 계산해 모아 둔다.
3. 네 지표의 평균을 그 K 의 한 행으로 담는다.
4. 행들을 DataFrame 으로 만든다.
```

</details>

In [ ]:
bs_rows = []
for k in (1, 2, 3, 5):
    hits, precs, recs, mrrs = [], [], [], []
    for row in bs_eval.itertuples():
        # 문항마다 검색은 한 번만 한다 - 지표마다 다시 검색하면 느리고, 재는 대상도 흔들린다.
        ranked = bs_search_ids(row.query, k)
        gold = row.gold_chunks.split('|')
        hits.append(hit_at_k(ranked, gold, k))
        precs.append(precision_at_k(ranked, gold, k))
        recs.append(recall_at_k(ranked, gold, k))
        mrrs.append(mrr_at_k(ranked, gold, k))
    bs_rows.append({'K': k,
                    'Hit': sum(hits) / len(hits),
                    'P': sum(precs) / len(precs),
                    'R': sum(recs) / len(recs),
                    'MRR': sum(mrrs) / len(mrrs)})

bs_k_table = pd.DataFrame(bs_rows)
display(bs_k_table.round(3))

In [ ]:
# [자가채점]
assert list(bs_k_table.columns) == ['K', 'Hit', 'P', 'R', 'MRR'], '열 이름과 순서를 맞춰 주세요'
assert list(bs_k_table['K']) == [1, 2, 3, 5]
# 재현율은 K 가 커질수록 줄어들 수 없고, 정밀도는 커질수록 늘어날 수 없다(구조적 성질)
assert list(bs_k_table['R']) == sorted(bs_k_table['R'])
assert list(bs_k_table['P']) == sorted(bs_k_table['P'], reverse=True)
# 실측값과 대조 - 검색은 결정적이라 같은 색인에서 같은 값이 나온다
assert abs(bs_k_table.loc[0, 'Hit'] - 0.857) < 0.02, 'K=1 의 Hit 이 실측과 다릅니다'
assert abs(bs_k_table.loc[0, 'R'] - 0.643) < 0.02, 'K=1 의 Recall 이 실측과 다릅니다'
assert abs(bs_k_table.loc[1, 'R'] - 0.929) < 0.02, 'K=2 의 Recall 이 실측과 다릅니다'
assert abs(bs_k_table.loc[3, 'P'] - 0.271) < 0.02, 'K=5 의 Precision 이 실측과 다릅니다'
print('✅ 통과!')

### 2단계 — 규칙에 따라 `k` 를 고르기

이제 정합니다. 우리 서비스의 기준은 이렇습니다.

> **답에 필요한 근거를 되도록 다 건지되(재현율 0.9 이상), 관련 없는 글은 되도록 적게 넣는다.**

- 이 기준을 만족하는 **가장 작은 K** 를 변수 **`chosen_k`** 에 **정수**로 담으세요.
- 왜 그 값인지 근거가 되는 두 수 — 그 K 의 재현율과 정밀도 — 를 각각 **`chosen_recall`**, **`chosen_precision`** 에 담으세요(실수).
- 세 값 모두 `bs_k_table` 에서 **찾아내야** 합니다. 눈으로 보고 손으로 적으면 안 됩니다.

<details><summary>힌트</summary>

```text
접근방법:
- 조건을 만족하는 행만 남기고, 그중 K 가 가장 작은 행을 고른다.

세부구현:
1. 재현율이 기준 이상인 행만 걸러 낸다.
2. 남은 행을 K 오름차순으로 보고 첫 행을 고른다.
3. 그 행에서 K 와 두 지표를 꺼내 각각 담는다.
   3-1. K 는 정수여야 한다.
```

</details>

In [ ]:
# 조건을 만족하는 행만 남긴다 - 손으로 고르지 않고 표에서 찾아내는 것이 핵심이다.
ok = bs_k_table[bs_k_table['R'] >= 0.9].sort_values('K')
best = ok.iloc[0]

chosen_k = int(best['K'])
chosen_recall = float(best['R'])
chosen_precision = float(best['P'])

print(f'고른 K: {chosen_k}')
print(f'  재현율 {chosen_recall:.3f} (기준 0.9 이상)')
print(f'  정밀도 {chosen_precision:.3f}')

In [ ]:
# [자가채점]
assert isinstance(chosen_k, int), 'chosen_k 는 정수여야 합니다'
assert chosen_k == 2, '기준을 만족하는 가장 작은 K 를 다시 확인하세요'
# 손으로 적은 값이 아니라 bs_k_table 에서 꺼낸 값인지 대조한다
chosen_row = bs_k_table[bs_k_table['K'] == chosen_k].iloc[0]
assert abs(chosen_recall - chosen_row['R']) < 1e-9, 'chosen_recall 은 bs_k_table 에서 꺼내세요'
assert abs(chosen_precision - chosen_row['P']) < 1e-9, 'chosen_precision 은 bs_k_table 에서 꺼내세요'
assert chosen_recall >= 0.9, '고른 K 가 기준을 만족하지 않습니다'
# 더 작은 K 는 기준을 만족하지 않아야 한다(가장 작은 K 여야 하므로)
smaller_rows = bs_k_table[bs_k_table['K'] < chosen_k]
assert (smaller_rows['R'] < 0.9).all(), '더 작은 K 로도 기준을 만족합니다 - 다시 고르세요'
print('✅ 통과!')

### 3단계 — 못 찾은 문항 읽기 (서술형)

평균만 보고 끝내지 않습니다. **`chosen_k`** 로 쟀을 때 **`Hit` 이 0 인 문항**을 찾아 출력하세요 — 질문·정답 라벨·실제 검색 결과를 나란히 봅니다. 변수 이름은 **`missed`**(문항 id 의 리스트)로 하세요. 적중 여부는 1번에서 만든 **`hit_at_k`** 로 판단합니다(`0.0` 이면 못 찾은 문항입니다).

그 문항을 실제로 읽고, **아래 markdown 셀에** 두 가지를 적으세요.

1. 왜 못 찾았다고 생각하나요? (질문의 낱말과 정답 FAQ 의 낱말을 견주어 보세요)
2. `K` 를 더 키우면 이 문항이 해결될까요? 표를 근거로 답하세요.

<details><summary>힌트</summary>

```text
접근방법:
- 문항마다 고른 K 로 검색해 정답이 하나도 안 들어왔는지 본다.

세부구현:
1. 평가셋을 돌며 고른 K 로 검색한다.
2. 정답 목록과 겹치는 것이 하나도 없으면 그 문항 id 를 모은다.
3. 모은 문항의 질문·정답·검색 결과를 차례로 출력한다.
```

</details>

In [ ]:
missed = []
for row in bs_eval.itertuples():
    ranked = bs_search_ids(row.query, chosen_k)
    gold = row.gold_chunks.split('|')
    if hit_at_k(ranked, gold, chosen_k) == 0.0:
        missed.append(row.query_id)
        print(f'[{row.query_id}] {row.query}')
        print('  정답:', gold)
        print('  검색:', ranked)

print('못 찾은 문항:', missed)

In [ ]:
# [자가채점]
assert isinstance(missed, list) and len(missed) == 1, '고른 K 에서 못 찾은 문항은 하나입니다'
# missed 가 손으로 적은 값이 아니라 실제로 재서 나온 값인지 대조한다
want_missed = [r.query_id for r in bs_eval.itertuples()
         if hit_at_k(bs_search_ids(r.query, chosen_k), r.gold_chunks.split('|'), chosen_k) == 0.0]
assert missed == want_missed, 'missed 는 실제로 재서 모은 목록이어야 합니다'
print('✅ 통과!')

**모범 서술**

**1. 왜 못 찾았나** — 질문은 "책이 아직 안 왔는데 어디에서 확인하나요" 인데, 답이 있는 FAQ 의 제목은 **배송 안내**이고 본문에서 그 답에 해당하는 부분은 "배송 조회는 마이페이지의 주문 내역에서 확인할 수 있습니다" 한 문장뿐입니다. 질문에는 **배송**이라는 낱말이 아예 없고, "확인"·"어디" 같은 말은 도서 검색·품절 안내 쪽 글과도 잘 어울립니다. 벡터 검색은 **글 전체의 의미**로 가까움을 재는데, 이 FAQ 는 배송 기간·배송비 이야기가 대부분이라 질문과 겹치는 부분이 묻혔습니다. **정답 문장은 있지만 그 문장이 글 전체에서 차지하는 비중이 작을 때** 생기는 전형적인 실패입니다.

**2. K 를 키우면 해결되나** — **안 됩니다.** 표에서 K=3 과 K=5 의 `Hit` 이 K=2 와 같은 0.929 로 **그대로**입니다. 상위 5개까지 넓혀도 이 문항의 정답은 올라오지 않습니다. 반면 정밀도는 0.679 에서 0.271 로 크게 떨어집니다. **K 조정으로는 이 문제를 해결할 수 없습니다** — 자르는 크기를 바꾸거나 질문을 다시 쓰거나 검색 방식 자체를 바꿔야 하는 문제입니다.

---
수고했어요! 검색을 재는 **눈금 네 개**를 손으로 만들고, 그것으로 서로 다른 두 검색기를 재어 **쓸 `k` 를 근거를 가지고** 골랐습니다. 이 과제에서는 모델을 한 번도 부르지 않았지요 — **판단의 근거를 수치로 만드는 일**에는 모델이 필요 없습니다. 평가셋을 **어떻게 만드는지**가 궁금하다면 `교안_03_평가셋_구축.ipynb` 를 보세요.